<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-27 16:03:36
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44


=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.54 C
CAGR/XIRR %:     1.42%
Est FTT Value:   2.74 C
Future Upside:   1.19 C (77.35%)
=================== PERFORMANCE ==========================
Current PnL:             -40.94 L (-21.33%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    8.28 L (5.56%)
CY Booked PnL:           6.36 L (4.12%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               -1.08 L (-0.69%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             2.88 L
Total Loss:               -43.83 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,UNITDSPR,-0.83,8.44,11.37,20.77,28869.0,19762.0,253906.0,1644.00,-5.05,61.0,X-MC,3.94,64.0,0.68,1.65,22.05,X40N,NTT,BREWERIES
50,M&M,-1.08,7.71,9.07,17.48,29414.0,23225.0,324300.0,3762.88,-4.66,64.0,X-LC,6.72,25.0,0.79,2.10,25.40,X40N,ATH,AUTO
12,BOSCH-HCIL,-0.38,-24.45,32.35,-0.01,31236.0,-31250.0,96555.0,2282.00,754.27,47.0,M-SC,17.91,231.0,-1.00,0.63,64.35,OX40N,BTT,AC
75,TTKPRESTIG,1.75,-31.05,45.17,0.09,31385.0,-31295.0,69482.0,770.00,-30.33,45.0,M-SC,3.05,250.0,-1.00,0.45,25.06,OX40N,NTT,DURABLES
52,MASFIN,-0.80,-22.06,55.58,21.26,42442.0,-21618.0,76362.0,396.02,-39.94,42.0,H-SC,13.85,167.0,-0.51,0.49,4.04,XR,ATH,FINANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,-4.67,2.57,38.72,42.29,112625.0,7299.0,290870.0,2782.73,19.66,43.0,X-LC,4.15,29.0,0.06,1.89,19.73,X40N,ATH,AUTO
47,JIOFIN,-1.94,-21.72,73.85,36.09,232119.0,-87234.0,314311.0,387.00,-30.48,50.0,X-LC,7.13,34.0,-0.38,2.04,8.85,XY24,BTT,FINANCE
4,ASIANPAINT,-0.41,-7.63,35.13,24.83,115661.0,-27179.0,329238.0,3422.40,-18.61,54.0,X-LC,3.49,30.0,-0.23,2.13,20.71,X40,ATH,PAINTS
44,INFY,-2.68,-16.71,71.28,42.66,237988.0,-66978.0,333877.0,1972.00,-39.32,56.0,X-LC,5.19,3.0,-0.28,2.16,19.52,X40,NTT,IT
80,VBL,-0.82,-12.35,57.36,37.93,196928.0,-48367.0,343319.0,668.62,-29.90,50.0,X-LC,6.25,13.0,-0.25,2.23,11.67,X40N,ATH,FMCG


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
62,ROUTE,5.06,-67.06,400.17,64.76,175411.0,-89232.0,43834.0,2170.75,NaN,49.0,NaN,10.72,141.0,-0.51,0.28,6.58,SR,ATH,IT
74,TRIDENT,4.04,-36.60,102.70,28.51,60069.0,-33764.0,58490.0,48.00,-30.37,58.0,M-SC,5.69,223.0,-0.56,0.38,15.57,XR,NTT,TEXTILES
22,DIXON,3.30,-4.29,43.99,37.82,109809.0,-11176.0,249622.0,18917.17,-54.85,56.0,X-MC,3.33,54.0,-0.10,1.62,35.93,X40N,ATH,IT
25,FINCABLES,2.68,49.90,37.72,106.44,83480.0,73670.0,221314.0,1629.94,-6.40,46.0,M-SC,19.49,220.0,0.88,1.43,67.69,OX40N,ATH,CABLES
6,AWL,2.29,-41.71,186.52,67.02,387391.0,-148615.0,207694.0,485.00,-74.37,48.0,X-SC,10.72,81.0,-0.38,1.35,3.31,XY24,NTT,FMCG


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,-4.67,2.57,38.72,42.29,112625.0,7299.0,290870.0,2782.73,19.66,43.0,X-LC,4.15,29.0,0.06,1.89,19.73,X40N,ATH,AUTO
24,EASEMYTRIP,-4.59,-66.72,337.81,45.70,198771.0,-117974.0,58841.0,26.40,NaN,51.0,NaN,16.01,204.0,-0.59,0.38,13.13,XY24,NTT,TRAVEL
32,HAVELLS,-3.27,-26.86,79.68,31.42,191156.0,-88087.0,239904.0,2052.62,-37.39,51.0,X-MC,9.50,63.0,-0.46,1.55,13.28,X40,ATH,ELECTRICAL
33,HCLTECH,-3.03,-2.55,29.00,25.70,102008.0,-9220.0,351753.0,1837.07,-9.13,58.0,X-LC,9.57,8.0,-0.09,2.28,40.34,X40,ATH,IT
28,GODREJCP,-3.02,-16.41,58.03,32.10,121045.0,-40947.0,208590.0,1452.16,-20.11,58.0,X-MC,12.84,32.0,-0.34,1.35,13.02,X40N,ATH,FMCG


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.68,49.90,37.72,106.44,83480.0,73670.0,221314.0,1629.94,-6.40,46.0,M-SC,19.49,220.0,0.88,1.43,67.69,OX40N,ATH,CABLES
65,SIS,0.60,-17.15,46.36,21.26,42647.0,-19041.0,91991.0,528.00,76.83,37.0,H-SC,4.70,168.0,-0.45,0.60,36.78,OX40N,NTT,MISC
12,BOSCH-HCIL,-0.38,-24.45,32.35,-0.01,31236.0,-31250.0,96555.0,2282.00,754.27,47.0,M-SC,17.91,231.0,-1.00,0.63,64.35,OX40N,BTT,AC
42,INDIGOPNTS,0.10,-30.39,43.69,0.02,53087.0,-53051.0,121508.0,1408.00,-33.68,49.0,M-SC,6.43,222.0,-1.00,0.79,38.44,OX40N,NTT,PAINTS
75,TTKPRESTIG,1.75,-31.05,45.17,0.09,31385.0,-31295.0,69482.0,770.00,-30.33,45.0,M-SC,3.05,250.0,-1.00,0.45,25.06,OX40N,NTT,DURABLES


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.68,49.9,37.72,106.44,83480.0,73670.0,221314.0,1629.94,-6.4,46.0,M-SC,19.49,220.0,0.88,1.43,67.69,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.68,49.90,37.72,106.44,83480.0,73670.0,221314.0,1629.94,-6.40,46.0,M-SC,19.49,220.0,0.88,1.43,67.69,OX40N,ATH,CABLES
66,STARHEALTH,1.00,10.39,33.40,47.26,103649.0,29202.0,310325.0,761.00,-19.26,56.0,H-SC,3.70,173.0,0.28,2.01,34.18,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.68,49.90,37.72,106.44,83480.0,73670.0,221314.0,1629.94,-6.40,46.0,M-SC,19.49,220.0,0.88,1.43,67.69,OX40N,ATH,CABLES
83,WIPRO,-2.26,-11.72,118.64,93.01,262094.0,-29325.0,220915.0,420.00,-23.98,55.0,M-LC,7.39,98.0,-0.11,1.43,22.52,XR,NTT,IT
69,TANLA,-1.22,-30.26,208.55,115.17,393778.0,-81944.0,188817.0,1903.93,-23.91,46.0,H-SC,27.16,147.0,-0.21,1.22,66.91,AR,ATH,IT
41,INDIAMART,0.40,-33.79,203.47,100.92,166147.0,-41679.0,81657.0,4675.60,-60.49,45.0,H-SC,16.54,140.0,-0.25,0.53,3.60,AR,ATH,MISC
3,ALKYLAMINE,2.28,-35.92,200.49,92.54,129705.0,-36269.0,64694.0,4520.85,-46.05,34.0,H-SC,7.77,142.0,-0.28,0.42,22.85,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
54,PAGEIND,0.03,-19.48,64.21,32.21,138751.0,-52289.0,216090.0,50690.45,-38.91,34.0,X-SC,8.30,55.0,-0.38,1.40,3.69,X40,ATH,APPARELS
37,HONAUT,0.34,-24.35,99.46,50.88,145087.0,-46963.0,145875.0,58192.80,-40.78,35.0,X-SC,10.03,90.0,-0.32,0.95,11.14,X40N,ATH,ELECTRICAL
73,TMPV,-1.53,-35.67,132.56,49.60,406644.0,-170110.0,306762.0,600.00,-525.11,39.0,X-MC,16.89,4.0,-0.42,1.99,2.16,XY24,NTT,AUTO
38,HYUNDAI,-4.67,2.57,38.72,42.29,112625.0,7299.0,290870.0,2782.73,19.66,43.0,X-LC,4.15,29.0,0.06,1.89,19.73,X40N,ATH,AUTO
10,BATAINDIA,-1.16,-59.73,235.71,35.20,122147.0,-76849.0,51821.0,2096.00,-19.49,44.0,X-SC,6.27,91.0,-0.63,0.34,7.69,X40,NTT,FOOTWEAR


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
1,ABBOTINDIA,-0.57,-8.94,26.69,15.37,43470.0,-15985.0,162870.0,34390.26,-26.93,58.0,X-MC,0.42,58.0,-0.37,1.06,10.33,X40,ATH,PHARMA
11,BERGEPAINT,1.57,-13.97,45.66,25.32,101253.0,-35995.0,221754.0,680.00,-23.06,48.0,X-MC,2.70,68.0,-0.36,1.44,16.29,XY24,NTT,PAINTS
17,CIPLA,-0.70,8.24,14.46,23.89,47055.0,24762.0,325417.0,1648.14,42.35,63.0,X-MC,3.15,18.0,0.53,2.11,21.89,X40N,ATH,PHARMA
22,DIXON,3.30,-4.29,43.99,37.82,109809.0,-11176.0,249622.0,18917.17,-54.85,56.0,X-MC,3.33,54.0,-0.10,1.62,35.93,X40N,ATH,IT
4,ASIANPAINT,-0.41,-7.63,35.13,24.83,115661.0,-27179.0,329238.0,3422.40,-18.61,54.0,X-LC,3.49,30.0,-0.23,2.13,20.71,X40,ATH,PAINTS


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
73,TMPV,-1.53,-35.67,132.56,49.60,406644.0,-170110.0,306762.0,600.00,-525.11,39.0,X-MC,16.89,4.0,-0.42,1.99,2.16,XY24,NTT,AUTO
6,AWL,2.29,-41.71,186.52,67.02,387391.0,-148615.0,207694.0,485.00,-74.37,48.0,X-SC,10.72,81.0,-0.38,1.35,3.31,XY24,NTT,FMCG
54,PAGEIND,0.03,-19.48,64.21,32.21,138751.0,-52289.0,216090.0,50690.45,-38.91,34.0,X-SC,8.30,55.0,-0.38,1.40,3.69,X40,ATH,APPARELS
26,GILLETTE,0.46,-14.42,50.37,28.69,109449.0,-36606.0,217290.0,10891.01,-38.76,51.0,X-SC,6.11,83.0,-0.33,1.41,4.99,X40,ATH,FMCG
55,PGHH,-1.20,-46.44,144.64,31.02,165399.0,-99168.0,114352.0,17484.41,-59.73,53.0,X-SC,20.67,57.0,-0.60,0.74,7.65,X40,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
10,BATAINDIA,-1.16,-59.73,235.71,35.20,122147.0,-76849.0,51821.0,2096.00,-19.49,44.0,X-SC,6.27,91.0,-0.63,0.34,7.69,X40,NTT,FOOTWEAR
55,PGHH,-1.20,-46.44,144.64,31.02,165399.0,-99168.0,114352.0,17484.41,-59.73,53.0,X-SC,20.67,57.0,-0.60,0.74,7.65,X40,ATH,FMCG
37,HONAUT,0.34,-24.35,99.46,50.88,145087.0,-46963.0,145875.0,58192.80,-40.78,35.0,X-SC,10.03,90.0,-0.32,0.95,11.14,X40N,ATH,ELECTRICAL
2,ACC,-1.03,-44.03,212.33,74.81,318648.0,-118056.0,150072.0,3906.00,19.71,57.0,X-SC,10.73,85.0,-0.37,0.97,11.17,XY24,BTT,CEMENT
1,ABBOTINDIA,-0.57,-8.94,26.69,15.37,43470.0,-15985.0,162870.0,34390.26,-26.93,58.0,X-MC,0.42,58.0,-0.37,1.06,10.33,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,-0.75,-13.24,48.34,28.70,183526.0,-57939.0,379656.0,452.00,-17.95,70.0,X-LC,5.19,1.0,-0.32,2.46,19.49,X40,NTT,FMCG
71,TCS,-2.57,-22.52,63.59,26.75,240471.0,-109917.0,378158.0,4208.34,-38.21,63.0,X-LC,6.14,2.0,-0.46,2.45,31.16,X40,ATH,IT
44,INFY,-2.68,-16.71,71.28,42.66,237988.0,-66978.0,333877.0,1972.00,-39.32,56.0,X-LC,5.19,3.0,-0.28,2.16,19.52,X40,NTT,IT
73,TMPV,-1.53,-35.67,132.56,49.60,406644.0,-170110.0,306762.0,600.00,-525.11,39.0,X-MC,16.89,4.0,-0.42,1.99,2.16,XY24,NTT,AUTO
33,HCLTECH,-3.03,-2.55,29.00,25.70,102008.0,-9220.0,351753.0,1837.07,-9.13,58.0,X-LC,9.57,8.0,-0.09,2.28,40.34,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,-0.38,-24.45,32.35,-0.01,31236.0,-31250.0,96555.0,2282.00,754.27,47.0,M-SC,17.91,231.0,-1.00,0.63,64.35,OX40N,BTT,AC
79,VALIANTORG,-1.10,-31.51,154.29,74.15,172877.0,-51558.0,112047.0,838.00,NaN,37.0,NaN,32.60,145.0,-0.30,0.73,66.74,XR,NTT,CHEMICALS
69,TANLA,-1.22,-30.26,208.55,115.17,393778.0,-81944.0,188817.0,1903.93,-23.91,46.0,H-SC,27.16,147.0,-0.21,1.22,66.91,AR,ATH,IT
25,FINCABLES,2.68,49.90,37.72,106.44,83480.0,73670.0,221314.0,1629.94,-6.40,46.0,M-SC,19.49,220.0,0.88,1.43,67.69,OX40N,ATH,CABLES
27,GLAXO,-0.34,11.21,22.01,35.69,48781.0,22348.0,221632.0,3380.16,-19.30,61.0,X-MC,5.50,56.0,0.46,1.44,31.72,X40,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
34,HDFCBANK,-0.55,7.62,25.17,34.70,97960.0,27540.0,389192.0,1073.00,-17.62,71.0,X-LC,5.96,26.0,0.28,2.52,26.03,X40,BTT,BANKS
81,VOLTAS,-2.36,-7.16,56.39,45.20,100385.0,-13722.0,178020.0,1856.00,1.30,60.0,X-MC,14.57,79.0,-0.14,1.15,25.43,XY25,NTT,AC
61,REPCOHOME,-0.35,-19.42,135.58,89.82,312992.0,-55651.0,230854.0,880.00,NaN,61.0,NaN,7.56,161.0,-0.18,1.50,24.62,XY24,NTT,FINANCE
50,M&M,-1.08,7.71,9.07,17.48,29414.0,23225.0,324300.0,3762.88,-4.66,64.0,X-LC,6.72,25.0,0.79,2.10,25.40,X40N,ATH,AUTO
51,MARUTI,-1.78,1.30,24.37,25.99,83863.0,4415.0,344125.0,17120.06,4.65,67.0,X-LC,7.72,33.0,0.05,2.23,23.23,XR,ATH,AUTO


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,23.38
1,25,52.17
2,50,82.72


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    39.59
SC    28.02
MC    21.67
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.69
XY24     22.62
X40N     19.31
XY25     10.23
XR        9.80
OX40N     6.61
AR        3.07
SR        0.70
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    38.16
X-MC    20.27
H-SC    13.77
X-SC     8.74
M-SC     5.51
M-LC     1.43
M-MC     1.05
L-MC     0.35
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.86,-21.13,64.82
IT,12.71,-30.66,98.74
FINANCE,10.91,-21.13,72.68
AUTO,9.09,-14.03,55.93
PAINTS,5.50,-24.65,47.04
MISC,5.50,-45.83,104.44
INSURANCE,5.48,2.45,34.21
BANKS,5.28,-9.88,64.98
PHARMA,4.61,4.38,19.62


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4007151.0,19
X40,2057611.0,16
XR,1592063.0,13
X40N,1457783.0,12
AR,1033882.0,6
OX40N,771435.0,10
XY25,710791.0,6
SR,305116.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2361831.0,17
H-SC,2319751.0,15
X-MC,1840590.0,14
X-SC,1456176.0,8
M-SC,521669.0,7
M-LC,262094.0,1
M-MC,83021.0,1
L-MC,47970.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,,sum,count
Conviction,Criteria,,
X-LC,X40,1185846.0,8
H-SC,XY24,991562.0,5
X-SC,XY24,706039.0,2
X-MC,XY24,664181.0,3
H-SC,AR,559925.0,2
X-MC,X40N,554208.0,5
X-SC,X40,535746.0,4
X-LC,X40N,518415.0,4
H-SC,XR,391477.0,4


In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 96.0 seconds
